# Project 1

Welcome to the first NLP1 project. The goal of the first tutorial was to get you familiar with text segmentation and simple text classification for topic detection.
Given the skills you have developed in the first week, you will now be applying them within a different dataset and task.

We are recommending a set of tasks and associated datasets; using those, you will be experimenting on problems introduced during the tutorials. You will be applying hands-on the experiments you have already explored in the last tutorial, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and analysing errors. 

The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

As the tutorials will be building upon your knowledge throughout the weeks, the task and dataset you choose now will be the ones you use in projects 2 and 3; and the ones you will be finally writing about in your final blog post.  

## Get to work!

The next section has a checklist to help you not lose track of what needs to be done. Here we have a more informal guide, with prompts and questions for you. 


1. [Choose a dataset](http://nlp-illc.github.io/nlp1/). Spend some time understanding how the dataset looks and some statistics of it (e.g. how many instances of each classification label we have?)
2. Remember what we learnt during the tutorial: to develop any classifier of the kind we saw, we needed a way to segment text. Do you want to use the tokeniser we have developed? Do you need to train a new one? If you do, what hyperparameters shall you use? What data shall you use to train it? Consider that your dataset may be too small to train a tokeniser! Think whether you will need a larger resource of similar nature to train a tokeniser.
3. Start thinking how to represent your documents into features. We have experimented with token occurences and frequencies. Can you think of any other ways, possibly? 
4. We had experimented with Logistic Regression and a shallow neural network. You can experiment with and assess different architectures now and their hyper parameters (e.g. add layers, possibly change the activation functions, drop out rates).
5. Evaluate your models! Was the evaluation method adequate for your new experimental setup? Do you want to assess other aspects of performance?
6. Analyse the various models you assessed and try to decide on which is the best.

Be mindful that you will be using your experiments to write up your final blog, so make sure they are clean and neat!

You can write your own code and you can reuse code from the tutorial and its packages or from other sources. You may use coding assistants if you like, but use it mindfully as you take full responsibility for this kind of (semi-)automated coding. Also, keep a record of the kinds of things you delegate to any form of assistant, as you will be asked to disclose this in your blog post (not because we will pass a value judgement on it, but because it is a widespread and good practice to document such things).

You may use jupyter notebooks or python files, it's up to you as we will not directly assess your code. But, once again, be organised, as we will ask you to share a reproducible version of your experiments along with your blog post (again, because this is how real research is publicised). 

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

## Checklist

What **we expect you to do**:

- [ ] Compare different systems of text classification for your choice of data/task.
- Your comparisons should cover:
    - [ ] the two representation spaces in the tutorial (token indicators and bag-of-words) and an additional one of your choice.
    - [ ] the two kinds of architectures in the tutorial (log-linear and single-hidden-layer MLP) and an additional one of your choice (such as a modification of the MLP).
- [ ] Your comparisons must be based on cross-validation and a relevant metric (accuracy or else).
- [ ] You should make use of Bayesian estimation to help you analyse results in relation to a region of practical equivalence (ROPE), and you can motivate the ROPE size yourself.

What you need to **bring to the oral assessment**:
* You will be asked questions about the points above, you can bring a slide deck with up to five slides (one for each of the following topics _data and metrics_, _features_, _architectures_, _results_, _extra_); you _will not_ give a presentation, rather, these slides are to be thought of as "backup slides" that you can use to guide your answers to the questions the TA will ask you.
* You may also be asked questions about the tutorial, for those, you don't need to prepare any deliverable (a good way to prepare is to work through the tutorial and reflect on the questions there).

What you should **keep in mind** given that you will eventually write a blog post about your work throughout the course:
* Organise your code;
* Summarise your motivations and findings;
* Document your sources and any form of AI-assistance you use.

In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import part1
from datasets import load_dataset

dataset = load_dataset("google-research-datasets/paws", "labeled_final")
paws_outcome_space = part1.clslib.NamedClasses(['not paraphrase', 'paraphrase'])


## WE use the PAWS dataset
paws consist of a dataset of two sentences with a binary classsification with the question if two of the sentences are a pharaphrase of the other.


# Dataset statistics and examples

In [3]:
#An example of a sentence pair from our training corpus
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Sentence 2:", dataset["train"]["sentence2"][0])
print("Gold label (id):", dataset["train"]["label"][0])

Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Sentence 2: In October 1560 , he secretly met with the English ambassador , Nicolas Throckmorton , in Paris , and asked him for a passport to return to Scotland through England .
Gold label (id): 0


In [4]:
from collections import Counter
from statistics import mean, median

def count_labels(data):
    """
    count the number of data points for weach of the binary labels in the dataset
    """
    label_counts = Counter(data["label"])
    return label_counts

def print_precentages(label_counts, total):
    """
    print the percentage of each label in the dataset
    """
    for label, count in label_counts.items():
        percentage = (count / total) * 100
        print(f"Label {label}: {percentage }%")

def entence_length_stats(data):
    """
    compute the mean and median length of the sentences in the dataset
    """
    sentence_lengths = [(len(s1.split()), len(s2.split()), label) for s1, s2, label in zip(data["sentence1"], data["sentence2"], data["label"])]
    mean_length_s1 = mean(length[0] for length in sentence_lengths)
    mean_length_s2 = mean(length[1] for length in sentence_lengths)
    median_length_s1 = median(length[0] for length in sentence_lengths)
    median_length_s2 = median(length[1] for length in sentence_lengths)
    print(f"Mean length of sentence 1 (based on spaces): {mean_length_s1}")
    print(f"Mean length of sentence 2 (based on spaces): {mean_length_s2}")
    print(f"Median length of sentence 1 (based on spaces): {median_length_s1}")
    print(f"Median length of sentence 2 (based on spaces): {median_length_s2}")
    return mean_length_s1, mean_length_s2, median_length_s1, median_length_s2

print("Training set label counts:")
train_label_counts = count_labels(dataset["train"])
print_precentages(train_label_counts, len(dataset["train"]))
_ = entence_length_stats(dataset["train"])

Training set label counts:
Label 0: 55.81263537175361%
Label 1: 44.18736462824639%
Mean length of sentence 1 (based on spaces): 21.340782575251513
Mean length of sentence 2 (based on spaces): 21.337948624521772
Median length of sentence 1 (based on spaces): 21
Median length of sentence 2 (based on spaces): 21


In [5]:
import part1

In [6]:
# PAWS has no "text" column: use both sentences of every pair as the corpus
corpus = list(dataset["train"]["sentence1"]) 
print("datapoints: ", len(corpus))
print()


datapoints:  49401



In [7]:
vocab_size = 5000
min_frequency = 2
tokenizer = part1.experiment.train_tokenizer(corpus, vocab_size, min_frequency)

# tokenize the data
1) Token indicators (like the tutorial)
2) Bag of words (home made)
3) UNKOWN

In [8]:
#Token indicator code
import numpy as np  
from tokenizers import Tokenizer
class Encoder:

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        raise NotImplementedError("Concrete implementations will be discussed later")

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        raise NotImplementedError("Concrete implementations will be discussed later")
        
class TokenIndicator(Encoder):

    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer
        
    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        unique_token_ids = list(set(self.tokenizer.encode(x).ids))

        # A 0 vector with the same length as the vocabulary except in positions corresponding 
        # to tokens appearing in the document
        vector = np.zeros(len(self))
        vector[unique_token_ids] = 1
        return vector

data_token_indicator = TokenIndicator(tokenizer)


In [9]:
# bag of words representation
class BagOfWords(Encoder):
    def __init__(self, tokenizer: Tokenizer):
        self.tokenizer = tokenizer

    def __len__(self) -> int:
        """Dimensionality of the representation space"""
        return len(self.tokenizer.get_vocab())

    def __call__(self, x: str) -> 'np.array':
        """Represent x as a D-dimensional vector of real numbers"""
        encoded = self.tokenizer.encode(x)
        vector = np.zeros(len(self))
        for token_id in encoded.ids:
            vector[token_id] += 1
        return vector

data_bag_of_words = BagOfWords(tokenizer)

In [10]:
#example of both tkenizers
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Token Indicator representation:", data_token_indicator(dataset["train"]["sentence1"][0]))
print("total number of tokens in the sentence:", sum(data_token_indicator(dataset["train"]["sentence1"][0])))
#example of the representation of a sentence using the BagOfWords
print("Sentence 1:", dataset["train"]["sentence1"][0])
print("Bag of Words representation:", data_bag_of_words(dataset["train"]["sentence1"][0]))
print("total number of tokens in the sentence:", sum(data_bag_of_words(dataset["train"]["sentence1"][0])))

Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Token Indicator representation: [0. 0. 0. ... 0. 0. 0.]
total number of tokens in the sentence: 37.0
Sentence 1: In Paris , in October 1560 , he secretly met the English ambassador , Nicolas Throckmorton , asking him for a passport to return to England through Scotland .
Bag of Words representation: [0. 0. 0. ... 0. 0. 0.]
total number of tokens in the sentence: 41.0


# Train calssiefieres
1) Log linear like in the tutorial
2) the Mlp


In [11]:
import part1
my_seeds = [0] # [0, 3, 5] # add more seeds here for multiple runs of cross validation, eg. [0, 3, 5]  
num_folds = 5  # more makes each cross validation run more statistically informative, but it can get slow
my_cross_validation_sets = []

for seed in my_seeds:  
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            # we pair the texts with their labels
            pairs=list(zip(dataset["train"]["sentence1"], dataset["train"]["sentence2"], dataset["train"]["label"])), 
            
            k=num_folds,
            seed=seed
        )
    )

# loggistic 
WE have to rebuild the logistic regression from the tutorial since we have two input sentences so the fit is not correct. We work our own magic on it to make it work for two sentences.

WE have the adapted dunction from the turorial, we concate the two encoded verctors so it can be used as input.

In [12]:
from scipy.sparse import csr_matrix
from sklearn.linear_model import LogisticRegression
import pickle
from tqdm.auto import tqdm

def load(path):
    """
    Load a logistic model from {path}.pickle and return it.
    """
    with open(f"{path}.pickle", 'rb') as f:
        logistic_model = pickle.load(f)
    return logistic_model

def fit(training_set, encoder: 'Encoder', save_to=None, max_iter=100, n_jobs=-1):
    """
    Constructs, trains and returns a softmax regressor.
    
    training_set: list of text-label pairs (a text is a string, a label is a 0-based integer)
    encoder: an instance of the Encoder class (such as TokenIndicator)    
    save_to: (str) if given, save the model to {save_to}.pickle
    """
    # Convert the training and validation sets to vectors
    # we get two senteces for each data point, so we make it a vector [sentence1, sentence2]
    training_vectors = [np.concatenate([encoder(text[0]), encoder(text[1])]) for text in training_set]
    
    training_labels = [specific_row["label"] for specific_row in training_set]

    #To train the model more efficiently, we transform the training vectors to a sparse matrix
    X_train_sparse = csr_matrix(training_vectors)

    # Train the logistic model
    logistic_model = LogisticRegression(solver="saga", max_iter=max_iter, n_jobs=n_jobs) #sparse matrix frinedly solver 
    logistic_model.fit(X_train_sparse, training_labels)

    if save_to is not None:        
        with open(f"{save_to}.pickle", 'wb') as f:
            pickle.dump(logistic_model, f, protocol=pickle.HIGHEST_PROTOCOL)
        
    return logistic_model        

def predict(texts, model, encoder: 'Encoder'):
    """
    Predicts labels (via the most probable class decision rule) for a collection of texts. 
    (scikit-learn hard-codes the most probable class as the decision rule of choice)
    
    texts: a list of texts, each a string
    model: a trained instance of sklearn's LogisticRegression
    encoder: the Encoder used for in the model's training
    """
    validation_vectors = [encoder(text) for text in texts]
    X_val_sparse = csr_matrix(validation_vectors)
    predictions = model.predict(X_val_sparse)
    return predictions

def predict_probs(texts, model, encoder: 'Encoder'):
    """
    Predicts labels (via the most probable class decision rule) for a collection of texts. 
    (scikit-learn hard-codes the most probable class as the decision rule of choice)
    
    texts: a list of texts, each a string
    model: a trained instance of sklearn's LogisticRegression
    encoder: the Encoder used for in the model's training
    """
    validation_vectors = [encoder(text) for text in texts]
    X_val_sparse = csr_matrix(validation_vectors)
    probs = model.predict_proba(X_val_sparse)
    return probs


In [20]:
from sklearn.metrics import accuracy_score

fold_results = []
models = {}
#Cross validation is a list of dictionaries, where each dictionary contains the training and validation sets for a fold
def cross_validation_train_logistic(data_token_indicator, my_cross_validation_sets, num_folds):
    """
    Train a logistic regression model on each fold of the cross validation sets and evaluate it on the validation set.
    """
    for index, cross_validation in my_cross_validation_sets[0].items():
        print(f"Training fold {index + 1}/{num_folds}")

        model = fit(
            cross_validation["training"],
            data_token_indicator,
            save_to=None
        )
        models[index] = model

        validation_set = cross_validation["validation"]
        Validation = csr_matrix([
            np.concatenate([
                data_token_indicator(sentence1),
                data_token_indicator(sentence2)
            ])
            for sentence1, sentence2, label in validation_set
        ])
        y_validation = [label for _, _, label in validation_set]

        predictions = model.predict(Validation)
        accuracy = accuracy_score(y_validation, predictions)

        fold_results.append({
            "fold": index,
            "accuracy": accuracy
        })

        print(f"Validation accuracy: {accuracy:.4f}")

    print(f"\nMean accuracy: {np.mean([r['accuracy'] for r in fold_results]):.4f}")

cross_validation_train_logistic(data_token_indicator, my_cross_validation_sets, num_folds)

Training fold 1/5


TypeError: tuple indices must be integers or slices, not str

# MLP
THE same thing but now with an MLP
Ones again the data loader from the given model does not work with our dataswet ass such i change it


In [18]:
from scipy.sparse import csr_matrix
from torch.utils.data import Dataset
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm.auto import tqdm
import time
import scipy
import scipy.sparse
import json
import numpy as np

class SparseMLP(nn.Module):
    """One-hidden-layer MLP that accepts sparse input tensors.

    Inputs: feature vectors of size `input_dim` (e.g. bag-of-words).
    Process: sparse matmul into a dense hidden layer, ReLU + dropout, linear (num_classes) and log-softmax outputs
        log-softmax, rather than softmax, for numerical stability
    Output: logprobs over `num_classes`.
    """

    def __init__(self, input_dim, hidden_dim, num_classes, dropout=0.2):
        """
        input_dim: size of the input feature space (e.g. vocabulary size)
        hidden_dim: width of the hidden layer
        num_classes: number of output labels
        dropout: fraction of hidden units zeroed during training (regularisation)
        """
        super().__init__()

        # First-layer weights stay dense; inputs arriving here are sparse.
        self.weight = nn.Parameter(
            torch.empty(input_dim, hidden_dim)
        )
        self.bias = nn.Parameter(
            torch.zeros(hidden_dim)
        )

        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

        # Everything after the first layer is dense.
        self.output = nn.Sequential(
            nn.Linear(hidden_dim, num_classes),            
            nn.LogSoftmax(dim=-1) # for numerical stability, it is common to use log softmax, rather than softmax
        )

        nn.init.xavier_uniform_(self.weight) # why xavier?

    def forward(self, x):
        """Map a sparse batch [batch, input_dim] to class log-probabilities [batch, num_classes]."""
        # First matmul keeps x sparse; result is dense from here on.
        x = torch.sparse.mm(x, self.weight)
        x = x + self.bias

        x = self.activation(x)
        x = self.dropout(x)
        # logprobs
        return self.output(x)

def scipy_to_torch_sparse(x):
    """Convert a SciPy sparse matrix to a coalesced PyTorch sparse COO tensor.

    Input: SciPy sparse matrix.
    Output: torch sparse tensor with the same shape and non-zero values.
    """
    x = x.tocoo()
    indices = torch.tensor([x.row, x.col], dtype=torch.long)
    values = torch.tensor(x.data, dtype=torch.float32)

    return torch.sparse_coo_tensor(indices, values, size=x.shape).coalesce()

class SparseDataset(Dataset):
    """Wraps a SciPy sparse feature matrix and label list for use with DataLoader.

    Inputs: X (sparse matrix of examples), y (integer labels).
    Output per item: one sparse row and its label tensor.
    """

    def __init__(self, X, y):
        self.X = X
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def sparse_collate(batch):
    """Collate individual sparse rows into one PyTorch sparse batch.

    Input: list of (sparse row, label) pairs from SparseDataset.
    Process: stack SciPy rows, convert to a torch sparse tensor, stack labels.
    Output: (X_batch sparse tensor, y_batch long tensor).
    """
    X_rows, y_rows = zip(*batch)

    # Stack SciPy sparse rows, then convert the whole batch at once.
    X_batch = scipy.sparse.vstack(X_rows)
    X_batch = scipy_to_torch_sparse(X_batch)

    y_batch = torch.stack(y_rows)

    return X_batch, y_batch


def train_one_epoch(model, dataloader, optimizer, criterion, device="cpu"):
    """Run one training epoch: forward, loss, backward, optimizer step.

    Inputs: model, train dataloader, optimizer, loss function, device.
    Output: dict with average train loss and accuracy for the epoch.
    """
    if isinstance(device, str):
        device = torch.device(device)
    
    model.train()  # enable dropout / training behaviour

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for X_batch, y_batch in dataloader:
        # move the batch to the device
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()  # clear gradients from the previous step

        logprobs = model(X_batch) # forward pass
        loss = criterion(logprobs, y_batch) # compute the loss

        loss.backward()   # compute gradients
        optimizer.step()  # update parameters

        # accumulate weighted loss and correct predictions for epoch averages
        total_loss += loss.item() * y_batch.size(0)
        predictions = logprobs.argmax(dim=1) # get the predicted class
        total_correct += (predictions == y_batch).sum().item() # count the number of correct predictions
        total_examples += y_batch.size(0) # count the number of examples

    return {
        "loss": total_loss / total_examples, # average loss
        "accuracy": total_correct / total_examples, # average accuracy
    }

def evaluate(model, dataloader, criterion, device="cpu"):
    """Evaluate the model without updating weights.

    Inputs: model, eval dataloader, loss function, device.
    Output: (metrics dict with loss/accuracy, numpy array of predicted labels).
    """
    if isinstance(device, str):
        device = torch.device(device)
        
    model.eval()  # disable dropout / training behaviour

    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    all_preds = []
    with torch.no_grad():  # no gradients needed for evaluation

        for X_batch, y_batch in dataloader:
            # move the batch to the device
            X_batch = X_batch.to(device) 
            y_batch = y_batch.to(device)

            logprobs = model(X_batch) # forward pass

            loss = criterion(logprobs, y_batch) # compute the loss

            # accumulate metrics; also keep predictions for later use
            total_loss += loss.item() * y_batch.size(0)
            predictions = logprobs.argmax(dim=1) # get the predicted class
            all_preds.append(predictions)

            total_correct += (predictions == y_batch).sum().item() # count the number of correct predictions
            total_examples += y_batch.size(0)

    # concatenate batch predictions into one array
    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples
    }, torch.cat([t.detach().cpu() for t in all_preds]).numpy()

def train_neural_network_model(training_set, validation_set, num_classes, encoder, save_to,   
                                hidden_dim=128, dropout=0.1, batch_size=64, learning_rate=1e-3, 
                                weight_decay=1e-4, num_epochs=20, device=torch.device("cpu")):
    """Encode texts, train SparseMLP, and checkpoint the best validation model.

    Inputs: train/val (text, label) pairs, num_classes, encoder (text -> sparse vector), path prefix to save weights,
            plus standard training hyperparameters.
    Process: vectorise documents, build sparse dataloaders, train for `num_epochs`, save best-by-val-loss.
    Output: per-epoch metrics list, best epoch index, and validation predictions from the best model.
    """
    if isinstance(device, str):
        device = torch.device(device)
        
    time_start = time.time()
    # Convert the training and validation sets to vectors
    print("Transforming documents...")
    print(f"Training set size: {training_set[0][0:3]}")
    training_vectors = [np.concatenate([encoder(text[0]), encoder(text[1])]) for text in training_set]
    training_labels = [int(label[2]) for label in training_set]
    validation_vectors = [np.concatenate([encoder(text[0]), encoder(text[1])]) for text in validation_set]
    validation_labels = [int(label[2]) for label in validation_set]

    time_document_to_vector = time.time()
    print(f"Document transformation done! Time taken: {time_document_to_vector - time_start:.3f} seconds")
    
    # Keep SciPy sparse here; conversion to torch happens in sparse_collate
    X_train_sparse = csr_matrix(training_vectors)
    X_val_sparse = csr_matrix(validation_vectors)

    train_dataset = SparseDataset(X_train_sparse, training_labels) # create the training dataset
    val_dataset = SparseDataset(X_val_sparse, validation_labels) # create the validation dataset

    # create the data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=sparse_collate,
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=sparse_collate,
    )

    time_dataloader = time.time()
    print(f"Dataloader done! Time taken: {time_dataloader - time_document_to_vector:.3f} seconds")

    # save architecture-defining arguments
    with open(f"{save_to}.json", "w", encoding="utf-8") as f:
        args = {
            'input_dim': 2*len(encoder), 
            'hidden_dim': hidden_dim, 
            'num_classes': num_classes, 
            'dropout': dropout,
        }
        json.dump(args, f, indent=2, ensure_ascii=False)
        
    # Create the model based on the input dimension, chosen hidden dimension, number of classification classes and dropout rate
    model = SparseMLP(input_dim=2*len(encoder), hidden_dim=hidden_dim, num_classes=num_classes, dropout=dropout).to(device)
    criterion = nn.NLLLoss() # create the loss function
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay) # create the optimizer

    metrics = []
    print("Training...")
    predictions = None
    for epoch in tqdm(range(num_epochs)):
        # Train the model for one epoch
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
        # Evaluate the model on the validation set
        val_metrics, preds = evaluate(model, val_loader, criterion, device)
        # Store the metrics for this epoch
        metrics.append({"train_metrics": train_metrics, "val_metrics": val_metrics})

        print(
            f"Epoch {epoch + 1:02d} | "
            f"train loss: {train_metrics['loss']:.4f} | "
            f"train acc: {train_metrics['accuracy']:.4f} | "
            f"val loss: {val_metrics['loss']:.4f} | "
            f"val acc: {val_metrics['accuracy']:.4f}"
            )
        
        # Saving the best model according to validation loss
        if epoch != 0:
            if val_metrics["loss"] <= best_val_loss:
                best_val_loss = val_metrics["loss"]
                torch.save(model.state_dict(), f"{save_to}.pt")
                best_epoch = epoch
                predictions = preds
        else:
            best_val_loss = val_metrics["loss"]
            torch.save(model.state_dict(), f"{save_to}.pt")
            best_epoch = epoch
            predictions = preds
    
    time_training = time.time()
    print(f"Training done! Time taken: {time_training - time_dataloader:.3f} seconds")

    return metrics, best_epoch, predictions


def load(path, device="cpu"):
    """
    Reconstruct the model architecture (using {path}.json) and load trained weights (using {path}.pt) and return it.
    """

    with open(f"{path}.json", "r", encoding="utf-8") as f:
        args = json.load(f)
    
    with open(f"{path}.pt", "rb") as f:
        state_dict = torch.load(f)
        
    if isinstance(device, str):
        device = torch.device(device)
        
    model = SparseMLP(**args)
    model.load_state_dict(state_dict)
    model = model.to(device)
    return model


def predict_probs(texts, model, encoder, batch_size=10):
    """
    Return a numpy array with shape [num_texts, num_classes] with the pmfs of the input texts as predicted by the model.
    
    texts: list of texts, each a string
    model: trained model
    encoder: encoder used in training
    batch_size: how many at a time
    """
    
    device = model.weight.device
        
    vectors = [encoder(x) for x in texts]
    labels = [0 for _ in texts]

    x_sparse = csr_matrix(vectors)

    dl = DataLoader(
        SparseDataset(x_sparse, labels),
        batch_size=batch_size,
        shuffle=False,
        collate_fn=sparse_collate,
    )
    
    model.eval()  # disable dropout / training behaviour
    
    all_probs = []
    with torch.no_grad():  # no gradients needed for evaluation

        for x_batch, y_batch in dl:
            # move the batch to the device
            x_batch = x_batch.to(device) 
            y_batch = y_batch.to(device)
            logprobs = model(x_batch) # forward pass
            # accumulate metrics; also keep predictions for later use
            all_probs.extend(logprobs.exp().detach().cpu().numpy())
            
    return np.array(all_probs)

# def cross_validation_train_neural_network(encoder, my_cross_validation_sets, num_folds):
#     """
#     Train a neural network model on each fold of the cross validation sets and evaluate it on the validation set.
#     """
#     smlp_results = []
#     for index, cross_validation in my_cross_validation_sets[0].items():
#         print(f"Training fold {index + 1}/{num_folds}")

#         training_set = cross_validation["training"]
#         validation_set = cross_validation["validation"]
#         num_classes = len(paws_outcome_space)
    
#         model = train_neural_network_model(
#             training_set,
#             validation_set,
#             num_classes,
#             encoder,
#             save_to=None
#         )

#         y_validation = [label for _, _, label in validation_set]
#         predictions = model.predict(validation_set)
#         accuracy = accuracy_score(y_validation, predictions)

#         smlp_results.append({
#             "fold": index,
#             "accuracy": accuracy
#         })

#         print(f"Validation accuracy: {accuracy:.4f}")
#     return smlp_results

def cross_validation_train_neural_network(encoder, my_cross_validation_sets, num_folds):
    smlp_results = []

    for index, cross_validation in my_cross_validation_sets[0].items():
        print(f"Training fold {index + 1}/{num_folds}")

        training_set = cross_validation["training"]
        validation_set = cross_validation["validation"]

        metrics, best_epoch, predictions = train_neural_network_model(
            training_set=training_set,
            validation_set=validation_set,
            num_classes=len(paws_outcome_space),
            encoder=encoder,
            save_to=f"mlp_fold_{index}"
        )

        y_validation = [label for _, _, label in validation_set]
        accuracy = accuracy_score(y_validation, predictions)

        smlp_results.append({
            "fold": index,
            "accuracy": accuracy,
            "best_epoch": best_epoch
        })

        print(f"Validation accuracy: {accuracy:.4f}")

    return smlp_results


cross_validation_train_neural_network(
    data_bag_of_words,
    my_cross_validation_sets,
    num_folds
)



Training fold 1/5
Transforming documents...
Training set size: ('Bhati is a census in South District in the state of Delhi , India .', 'Bhati is a census town in South district in the state of Delhi , India .', 1)
Document transformation done! Time taken: 120.741 seconds
Dataloader done! Time taken: 6.979 seconds
Training...


  0%|          | 0/20 [00:00<?, ?it/s]

Epoch 01 | train loss: 0.6784 | train acc: 0.5772 | val loss: 0.6649 | val acc: 0.5934
Epoch 02 | train loss: 0.6516 | train acc: 0.6060 | val loss: 0.6611 | val acc: 0.5929
Epoch 03 | train loss: 0.6366 | train acc: 0.6204 | val loss: 0.6665 | val acc: 0.5782
Epoch 04 | train loss: 0.6265 | train acc: 0.6289 | val loss: 0.6879 | val acc: 0.5574
Epoch 05 | train loss: 0.6197 | train acc: 0.6355 | val loss: 0.6981 | val acc: 0.5566
Epoch 06 | train loss: 0.6127 | train acc: 0.6412 | val loss: 0.7232 | val acc: 0.5376
Epoch 07 | train loss: 0.6074 | train acc: 0.6431 | val loss: 0.7283 | val acc: 0.5417
Epoch 08 | train loss: 0.6010 | train acc: 0.6511 | val loss: 0.7548 | val acc: 0.5276
Epoch 09 | train loss: 0.5940 | train acc: 0.6559 | val loss: 0.7892 | val acc: 0.5060
Epoch 10 | train loss: 0.5882 | train acc: 0.6597 | val loss: 0.8070 | val acc: 0.5151
Epoch 11 | train loss: 0.5820 | train acc: 0.6651 | val loss: 0.8532 | val acc: 0.5012
Epoch 12 | train loss: 0.5758 | train acc: 

KeyboardInterrupt: 